Purpose
---
Aggregate the number of active Spanish vessels by base port and year.

Input:
    RGFP Excel workbook with one row per active vessel/year.

Output:
    boats_year_port.csv with columns:
        PUERTO BASE ; AÑO ; NUM_BARCOS

# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

INPUT_FILE = "/content/drive/MyDrive/BCKG/CSV Barcos/rgfp_bi_excel_2006_2025_20260127.xlsx"
OUTPUT_FILE = "/content/drive/MyDrive/BCKG/CSV Barcos/boats_year_port2.csv"

SHEET_NAME = "Datos"

REQUIRED_COLUMNS = [
    "AÑO",
    "CODIGOBUQUE",
    "PUERTO BASE",
]

# Helpers

In [ ]:
def load_rgfp_excel(path, sheet_name):
    """
    Load the RGFP Excel workbook.

    The workbook is read from Google Drive and loaded as a pandas dataframe.
    """
    return pd.read_excel(path, sheet_name=sheet_name)


def validate_columns(df, required_columns):
    """
    Ensure the input dataframe contains the columns required by the aggregation.
    """
    missing = [col for col in required_columns if col not in df.columns]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Available columns: {list(df.columns)}"
        )


def normalize_port_name(value):
    """
    Normalize a base port name.

    This trims leading/trailing whitespace and collapses repeated internal spaces.
    """
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    text = " ".join(text.split())

    return text if text else pd.NA


def prepare_vessel_rows(df):
    """
    Keep only the fields required for aggregation and normalize base port names.

    Each remaining row represents an active vessel in a given year.
    """
    validate_columns(df, REQUIRED_COLUMNS)

    prepared = df[REQUIRED_COLUMNS].copy()

    prepared["PUERTO BASE"] = prepared["PUERTO BASE"].apply(normalize_port_name)

    # Rows without a year or a base port cannot be aggregated reliably.
    prepared = prepared.dropna(subset=["AÑO", "PUERTO BASE"])

    return prepared


def aggregate_boats_by_port_year(df):
    """
    Count active vessels by base port and year.

    The source file is assumed to have one row per active vessel/year.
    """
    counts = (
        df.groupby(["PUERTO BASE", "AÑO"], as_index=False)
          .size()
          .rename(columns={"size": "NUM_BARCOS"})
    )

    return counts


def save_output(df, path):
    """
    Save the aggregated dataframe as a semicolon-separated CSV.
    """
    df.to_csv(path, sep=";", index=False, encoding="utf-8-sig")


# Pipeline execution

In [ ]:
df = load_rgfp_excel(INPUT_FILE, SHEET_NAME)
df = prepare_vessel_rows(df)
counts_long = aggregate_boats_by_port_year(df)

save_output(counts_long, OUTPUT_FILE)

print("Output generated in:")
print(OUTPUT_FILE)
print(f"Rows exported: {len(counts_long)}")